# Train Kaggle FER OpenVINO Model

This is the only notebook for training the new Kaggle facial-expression model. Use `Run All` here.

It prepares the raw YOLO dataset, runs CPU-friendly transfer learning with early YOLO layers frozen, evaluates it, exports OpenVINO, validates the OpenVINO export, and publishes the model for the realtime AI service.

Important: Ultralytics training still uses PyTorch internally. OpenVINO is used after training for export, validation, and realtime inference on Intel CPU/GPU/NPU.

In [1]:
from pathlib import Path
import json
import random
import shutil
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'kaggle_fer_yolo_raw'
PREPARED_DIR = PROJECT_ROOT / 'data' / 'fer_expression_yolo_prepared'
RUNS_DIR = PROJECT_ROOT / 'runs' / 'fer_expression'
PUBLISHED_MODEL_DIR = PROJECT_ROOT / 'models' / 'fer_expression_yolo26n_openvino'
OLD_MODEL_DIR = PROJECT_ROOT / 'fer_yolo26s_cls_balanced_safeaug_e100-5'

# CPU-friendly transfer learning: train only the last YOLO blocks, then export to OpenVINO.
BASE_MODEL = 'yolo26n-cls.pt'
RUN_NAME = 'fer_yolo26n_cls_kaggle_last_blocks'
IMG_SIZE = 224
EPOCHS = 20
BATCH = 16
PATIENCE = 5
# Ultralytics freezes layers [0, freeze-1]. For YOLO26n-cls this leaves the final blocks and head trainable.
FREEZE_LAYERS = 8
SEED = 42
CROP_MARGIN = 0.10
BALANCE_TRAIN = True
MAX_BALANCE_MULTIPLIER = 3

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('Raw dataset exists:', RAW_DIR.exists(), RAW_DIR)
print('Output model:', PUBLISHED_MODEL_DIR)

Project root: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1
Python: c:\ProgramData\anaconda3\python.exe
Raw dataset exists: True C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\data\kaggle_fer_yolo_raw
Output model: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\models\fer_expression_yolo26n_openvino


In [2]:
import torch
import openvino as ov
from ultralytics import YOLO

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Train device:', '0' if torch.cuda.is_available() else 'cpu')
print('OpenVINO:', ov.__version__)
print('OpenVINO devices:', ov.Core().available_devices)
print('Ultralytics import OK')

<frozen importlib.util>:209: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.


Torch: 2.10.0+cpu
CUDA available: False
Train device: cpu
OpenVINO: 2025.0.0-17942-1f68be9f594-releases/2025/0
OpenVINO devices: ['CPU', 'GPU']
Ultralytics import OK


## Clean Old Outputs

This removes old training outputs and old model artifacts so the new run cannot be mixed with the old notebook run. It does not delete `data/kaggle_fer_yolo_raw`.

In [3]:
def ensure_inside_project(path: Path) -> Path:
    resolved = path.resolve()
    if resolved != PROJECT_ROOT and PROJECT_ROOT not in resolved.parents:
        raise ValueError(f'Refusing to delete outside project: {resolved}')
    return resolved

for target in (PREPARED_DIR, RUNS_DIR, PUBLISHED_MODEL_DIR, OLD_MODEL_DIR):
    target = ensure_inside_project(target)
    if target.exists():
        shutil.rmtree(target)
        print('Removed:', target)
    else:
        print('Already clean:', target)

Already clean: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\data\fer_expression_yolo_prepared
Already clean: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\runs\fer_expression
Already clean: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\models\fer_expression_yolo26n_openvino
Already clean: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\fer_yolo26s_cls_balanced_safeaug_e100-5


## Prepare Dataset

The Kaggle dataset is YOLO detection format. This cell crops each labeled face box into classification folders that YOLO classification training can use.

In [ ]:
import cv2
import numpy as np
import yaml

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_ALIASES = {
    'train': ('train', 'training'),
    'val': ('val', 'valid', 'validation'),
    'test': ('test', 'testing'),
}

def normalize_label(label: str) -> str:
    cleaned = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(label)).strip('_')
    aliases = {
        'anger': 'angry',
        'happiness': 'happy',
        'natural': 'neutral',
        'sadness': 'sad',
        'surprised': 'surprise',
    }
    return aliases.get(cleaned, cleaned or 'unknown')

def image_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def read_image_bgr(path: Path):
    # cv2.imread can fail on Vietnamese/Unicode Windows paths; imdecode works reliably.
    data = np.fromfile(path, dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def write_jpg(path: Path, frame_bgr) -> bool:
    ok, encoded = cv2.imencode('.jpg', frame_bgr)
    if not ok:
        return False
    path.write_bytes(encoded.tobytes())
    return True

def read_yolo_rows(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    rows = []
    for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            rows.append((int(float(parts[0])), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))
    return rows

def box_to_pixels(xc: float, yc: float, bw: float, bh: float, width: int, height: int) -> tuple[int, int, int, int]:
    side = max(bw * width, bh * height) * (1.0 + 2.0 * CROP_MARGIN)
    cx = xc * width
    cy = yc * height
    x1 = max(0, int(round(cx - side / 2.0)))
    y1 = max(0, int(round(cy - side / 2.0)))
    x2 = min(width, int(round(cx + side / 2.0)))
    y2 = min(height, int(round(cy + side / 2.0)))
    return x1, y1, x2, y2

def find_dataset_root(raw_dir: Path) -> Path:
    candidates = [raw_dir] + [p for p in raw_dir.rglob('*') if p.is_dir()]
    scored = []
    for path in candidates:
        score = 0
        score += 10 if (path / 'data.yaml').exists() else 0
        score += 4 if (path / 'train' / 'images').exists() else 0
        score += 2 if (path / 'valid' / 'images').exists() or (path / 'val' / 'images').exists() else 0
        score += 1 if (path / 'test' / 'images').exists() else 0
        scored.append((score, path))
    score, path = max(scored, key=lambda item: item[0])
    if score <= 0:
        raise FileNotFoundError(f'No YOLO dataset found under {raw_dir}')
    return path

def split_image_dir(root: Path, split: str) -> Path | None:
    for alias in SPLIT_ALIASES[split]:
        for candidate in (root / alias / 'images', root / 'images' / alias, root / alias):
            if candidate.exists() and image_files(candidate):
                return candidate
    return None

def matching_label_dir(image_dir: Path) -> Path:
    parts = list(image_dir.parts)
    if 'images' in parts:
        parts[parts.index('images')] = 'labels'
        return Path(*parts)
    return image_dir.parent / 'labels'

DATASET_ROOT = find_dataset_root(RAW_DIR)
data_yaml = yaml.safe_load((DATASET_ROOT / 'data.yaml').read_text(encoding='utf-8'))
names = data_yaml.get('names', {})
CLASS_NAMES = {i: normalize_label(v) for i, v in enumerate(names)} if isinstance(names, list) else {int(k): normalize_label(v) for k, v in names.items()}

print('Dataset root:', DATASET_ROOT)
print('Classes:', CLASS_NAMES)

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
counts = {split: Counter() for split in ('train', 'val', 'test')}

for split in ('train', 'val', 'test'):
    image_dir = split_image_dir(DATASET_ROOT, split)
    if image_dir is None:
        print('Missing split:', split)
        continue
    label_dir = matching_label_dir(image_dir)
    images = image_files(image_dir)
    print(f'Preparing {split}: {len(images)} images')
    for index, image_path in enumerate(images, start=1):
        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue
        frame = read_image_bgr(image_path)
        if frame is None:
            continue
        height, width = frame.shape[:2]
        for box_index, row in enumerate(read_yolo_rows(label_path)):
            class_id, xc, yc, bw, bh = row
            label = CLASS_NAMES.get(class_id, str(class_id))
            x1, y1, x2, y2 = box_to_pixels(xc, yc, bw, bh, width, height)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            target_dir = PREPARED_DIR / split / label
            target_dir.mkdir(parents=True, exist_ok=True)
            if write_jpg(target_dir / f'{image_path.stem}_{box_index:02d}.jpg', crop):
                counts[split][label] += 1
        if index % 1000 == 0:
            print(f'  {split}: {index}/{len(images)}')

if BALANCE_TRAIN:
    train_counts = {p.name: len(image_files(p)) for p in (PREPARED_DIR / 'train').iterdir() if p.is_dir()}
    max_count = max(train_counts.values()) if train_counts else 0
    random.seed(SEED)
    for label, count in train_counts.items():
        class_dir = PREPARED_DIR / 'train' / label
        files = image_files(class_dir)
        target_count = min(max_count, len(files) * MAX_BALANCE_MULTIPLIER)
        for i in range(max(0, target_count - len(files))):
            source = random.choice(files)
            shutil.copy2(source, class_dir / f'{source.stem}_bal{i:04d}{source.suffix}')

final_counts = {}
for split in ('train', 'val', 'test'):
    split_dir = PREPARED_DIR / split
    final_counts[split] = {p.name: len(image_files(p)) for p in sorted(split_dir.iterdir()) if p.is_dir()} if split_dir.exists() else {}

summary = {
    'dataset_root': str(DATASET_ROOT),
    'prepared_dir': str(PREPARED_DIR),
    'class_names': sorted({label for split_counts in final_counts.values() for label in split_counts}),
    'counts': final_counts,
}
(PREPARED_DIR / 'dataset_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=True), encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=True))

Dataset root: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sản xuất DevOps, DataOps, MLOps\Lab1\data\kaggle_fer_yolo_raw\9 Facial Expressions you need
Classes: {0: 'angry', 1: 'contempt', 2: 'disgust', 3: 'fear', 4: 'happy', 5: 'neutral', 6: 'sad', 7: 'sleepy', 8: 'surprise'}
Preparing train: 64864 images
  train: 1000/64864
  train: 2000/64864
  train: 3000/64864
  train: 4000/64864
  train: 5000/64864
  train: 6000/64864
  train: 7000/64864
  train: 8000/64864
  train: 9000/64864
  train: 10000/64864
  train: 11000/64864
  train: 12000/64864
  train: 13000/64864
  train: 14000/64864
  train: 15000/64864
  train: 16000/64864
  train: 17000/64864
  train: 18000/64864
  train: 19000/64864
  train: 20000/64864
  train: 21000/64864
  train: 22000/64864
  train: 23000/64864
  train: 24000/64864
  train: 25000/64864
  train: 26000/64864
  train: 27000/64864
  train: 28000/64864
  train: 29000/64864
  train: 30000/64864
  train: 31000/64864
  train: 32000/64864
  train: 33000/648

: 

## Train, Evaluate, Export

In [ ]:
device = '0' if torch.cuda.is_available() else 'cpu'
print('Training on device:', device)

model = YOLO(BASE_MODEL)
train_results = model.train(
    data=str(PREPARED_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    optimizer='auto',
    device=device,
    workers=0,
    seed=SEED,
    deterministic=True,
    cos_lr=True,
    plots=True,
    freeze=FREEZE_LAYERS,
)

run_dir = Path(train_results.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
print('Run dir:', run_dir)
print('Best checkpoint:', best_pt, best_pt.exists())

best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(PREPARED_DIR), imgsz=IMG_SIZE, batch=BATCH, device=device)
metrics_payload = {}
for attr in ('top1', 'top5', 'fitness'):
    if hasattr(metrics, attr):
        try:
            metrics_payload[attr] = float(getattr(metrics, attr))
        except TypeError:
            pass
if hasattr(metrics, 'results_dict'):
    metrics_payload['results_dict'] = {str(k): float(v) if isinstance(v, (int, float)) else str(v) for k, v in metrics.results_dict.items()}
(run_dir / 'metrics_summary.json').write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=True), encoding='utf-8')
print('Metrics:', json.dumps(metrics_payload, indent=2, ensure_ascii=True))

export_path = best_model.export(format='openvino', imgsz=IMG_SIZE, batch=1, half=False, int8=False)
export_dir = Path(export_path)
if export_dir.is_file():
    export_dir = export_dir.parent

if PUBLISHED_MODEL_DIR.exists():
    shutil.rmtree(PUBLISHED_MODEL_DIR)
PUBLISHED_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(export_dir, PUBLISHED_MODEL_DIR)
print('Published OpenVINO model:', PUBLISHED_MODEL_DIR)

openvino_model = YOLO(str(PUBLISHED_MODEL_DIR))
openvino_metrics = openvino_model.val(data=str(PREPARED_DIR), imgsz=IMG_SIZE, batch=BATCH, device='intel:cpu')
print('OpenVINO validation metrics:', openvino_metrics)

Training on device: cpu
New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.13.9 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong sn xut DevOps, DataOps, MLOps\Lab1\data\fer_expression_yolo_prepared, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=8, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lr

## Verify AI Service Model

In [1]:
import numpy as np

from fer_realtime.config import DEFAULT_OPENVINO_MODEL_PATH, IMG_SIZE
from fer_realtime.model import OpenVINOExpressionClassifier

print('Runtime default OpenVINO path:', str(DEFAULT_OPENVINO_MODEL_PATH).encode('ascii', errors='backslashreplace').decode('ascii'))
classifier = OpenVINOExpressionClassifier(model_path=DEFAULT_OPENVINO_MODEL_PATH, device='AUTO')
dummy = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
prediction = classifier.predict_batch([dummy])[0]
print('Verification label:', prediction.label)
print('Verification top_k:', prediction.top_k)
print('Device:', prediction.device)

Runtime default OpenVINO path: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\AI trong s\u1ea3n xu\u1ea5t DevOps, DataOps, MLOps\Lab1\models\fer_expression_yolo26n_openvino
Verification label: fear
Verification top_k: [('fear', 0.20501063764095306), ('sleepy', 0.18315185606479645), ('neutral', 0.1378265619277954)]
Device: OpenVINO (CPU)
